# <B>LangChain Expression Language (LCEL)</B>

---
---

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from rich import print as rprint
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from IPython.display import display, Markdown
from llm_models import OPENAI_MODELS, GROQ_MODELS
 

def openai_chat(model, temperature = 0, max_tokens = 500, seed = 365):
    return ChatOpenAI(
        model = model,
        temperature= temperature,
        max_tokens = max_tokens,
        seed = seed
    )

def groq_chat(model, temperature = 0, max_tokens = 500, seed = 365):
    return ChatGroq(
        model= model,
        temperature=temperature,
        max_tokens=max_tokens,
        seed = seed
    )

chat = groq_chat(model=GROQ_MODELS["gpt_oss_20b"])

def markdown_display(content: str): 
    return display(Markdown(content))

list_parser_instruction = CommaSeparatedListOutputParser().get_format_instructions()

### Piping a prompt, model, and output parser

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([("human", "Can you suggest five {pet} names? \n" + list_parser_instruction)])


In [ ]:
rprint(chat_template)

In [ ]:
template_result = chat_template.invoke({"pet": "dog"})
rprint(template_result)

In [ ]:
chat_val = chat.invoke(template_result)

In [ ]:
rprint(chat_val)

In [ ]:
list_parser = CommaSeparatedListOutputParser()
parsed = list_parser.invoke(chat_val)
rprint(parsed)

In [ ]:
chain = chat_template | chat | list_parser
chain.invoke({"pet": "dog"})

---

### Batching

In [ ]:
chat_template = ChatPromptTemplate.from_messages([
    (
        "human",
        "I want to buy a car from {brand}. Can you suggest some models the people like the most."
    )
    ])

In [ ]:
chain = chat_template | chat
chain_val = chain.invoke({"brand": "tesla"})
batch_val = chain.batch([{"brand": "tesla"}, {"brand": "Honda"}, {"brand": "Bugatti"}])


In [ ]:
rprint(batch_val)

In [ ]:
%%time
for ai_msg in batch_val:
    display(Markdown(ai_msg.content))
    display(Markdown("---"))

---

### Streaming

In [ ]:
stream_val = chain.stream({"brand": "Pagani"})

In [ ]:
for val in stream_val:
    print(val.content, end="", flush=True)

---

### The Runnable and RunnableSequence classes

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chat = groq_chat(model=GROQ_MODELS["llama_3.1_8b"])
chat_template = ChatPromptTemplate.from_messages([("human", "Can you please list some sources for {industry} news")])

In [ ]:
chained = chat_template | chat
val = chained.invoke({"industry": "Stocks"})
type(chained)

---

### Piping chains and RunnablePassthrough class

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
# chat_tool = ChatPromptTemplate.from_messages([("human", "Can you suggest some tools a {job_title} will need for their role. \n" + list_parser_instruction)])
chat_tool = ChatPromptTemplate.from_messages([("human", "Can you suggest some tools a {job_title} will need for their role.")])
chat_strategy = ChatPromptTemplate.from_messages([("human", "Considering the {tools} above, can you suggest an effective way of learning to use them.")])

chat = groq_chat(model=GROQ_MODELS["llama_3.1_8b"])
 
string_parser = StrOutputParser() 
list_parser = CommaSeparatedListOutputParser()

In [ ]:
# chain_tools = chat_tool | chat | list_parser
chain_tools = chat_tool | chat | string_parser | {"tools": RunnablePassthrough()}
chain_strategy = chat_strategy | chat | string_parser

In [ ]:
combined_chain = chain_tools | chain_strategy
val = combined_chain.invoke({"job_title": "Mobile App developer"})

In [ ]:
markdown_display(val)

--- 

In [ ]:
longer_chain = (
    chat_tool | 
    chat | 
    string_parser | 
    {"tools": RunnablePassthrough()} |
    chat_strategy |
    chat | 
    string_parser
    )

In [ ]:
long_val = longer_chain.invoke({"job_title": "Data Engineer"})
markdown_display(long_val)

### Graphing Runnables

In [ ]:
longer_chain.get_graph().print_ascii()

---

### RunnableParallel

In [ ]:
chat_template_books = ChatPromptTemplate.from_messages([
    ("human", """ 
     Suggest three of the best intermediate-level
     {programming_language} books.
     Answer only by listing the books.
     """)
])

chat_template_projects = ChatPromptTemplate.from_messages([(
    "human", """ 
    Suggest three insteresting {programming_language} projects
    suitable for intermediate-level programmers.
    Answer only by listing the projects
    """
)])

# chat_template_books = ChatPromptTemplate.from_template(
#     '''
#     Suggest three of the best intermediate-level {programming language} books. 
#     Answer only by listing the books.
#     '''
# )

# chat_template_projects = ChatPromptTemplate.from_template(
#     '''
#     Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
#     Answer only by listing the projects.
#     '''
# )

string_parser = StrOutputParser()

chain_books = chat_template_books | chat | string_parser
new_chat = openai_chat(model=OPENAI_MODELS["gpt_4.1_mini"])
chain_projects  = chat_template_projects | new_chat | string_parser

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel({"books": chain_books, "projects": chain_projects})

In [ ]:
output = parallel.invoke({"programming_language": "Python"})
rprint(output)

In [ ]:
parallel.get_graph().print_ascii()

---

In [ ]:
%%time
chain_books.invoke({"programming_language": "Python"})

In [ ]:
%%time
chain_projects.invoke({"programming_language": "Python"})

In [ ]:
%%time
parallel.invoke({"programming_language": "Python"})

### Piping a RunnableParallel with other Runnables

In [ ]:
chat_template_time = ChatPromptTemplate.from_messages([(
    "human", 
    """ 
    I'm an intermediate level programmer.
    Consider the following literature.
    {books}
    Also, consider the following projects:
    {projects}
    Roughly how much time would it take me to complete the literature and the projects?
 """
)])

In [ ]:
chain1 = (
    RunnableParallel({"books": chain_books, "projects": chain_projects}) | 
    chat_template_time | 
    chat | 
    string_parser
    )

In [ ]:
val = chain1.invoke({"programming_language": "Javascript"})
markdown_display(val)

In [ ]:
chain1.get_graph().print_ascii()

In [ ]:
chain2 = (
    {"books": chain_books, "projects": chain_projects} | 
    chat_template_time | 
    chat | 
    string_parser
    )

val2 = chain2.invoke({"programming_language": "Ruby"})
markdown_display(val2)

In [ ]:
chain2.get_graph().print_ascii()

---

### RunnableLamda

In [90]:
from langchain_core.runnables import RunnableLambda

# Runnable Sequence 
sequence = RunnableLambda(lambda x: x+x) | RunnableLambda(lambda x: x*10)
rprint(sequence.invoke(2))
rprint(sequence.batch([1,2,3]))

# Runnable Parallel
parallel = RunnableLambda(lambda x: x*1) | {
    "PXL1":  RunnableLambda(lambda x: x*10),
    "PXL2": RunnableLambda(lambda x: x*100)
}

rprint(parallel.invoke(20))
rprint(sequence.batch([1,2,3]))

40

[20, 40, 60]

{'PXL1': 200, 'PXL2': 2000}

[20, 40, 60]

In [91]:
sequence.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


In [92]:
parallel.get_graph().print_ascii()

        +-------------+        
        | LambdaInput |        
        +-------------+        
               *               
               *               
               *               
          +--------+           
          | Lambda |           
          +--------+           
               *               
               *               
               *               
 +--------------------------+  
 | Parallel<PXL1,PXL2>Input |  
 +--------------------------+  
          *         *          
        **           **        
       *               *       
+--------+          +--------+ 
| Lambda |          | Lambda | 
+--------+          +--------+ 
          *         *          
           **     **           
             *   *             
+---------------------------+  
| Parallel<PXL1,PXL2>Output |  
+---------------------------+  


---

### The @chain decorator

In [93]:
from langchain_core.runnables import chain

@chain
def runnable_sum(x):
    return sum(x)
@chain
def runnable_squares(x):
    return x**2

In [94]:
print(type(runnable_sum))
print(type(runnable_squares))

<class 'langchain_core.runnables.base.RunnableLambda'>
<class 'langchain_core.runnables.base.RunnableLambda'>


In [98]:
chained = runnable_sum | runnable_squares
print(chained.invoke([1,2,3]))
print(chained.batch([[1,2,3], [2,3,4]]))

36
[36, 81]


---

---
---

# Concept Testing

---
---

In [ ]:
# def func1():
#     return "Richmond"

# def func2(name):
#     return f"Hi {name}"

# chained = func1 | func2
# print(chained)

class Runnable:
    def __init__(self, func):
        self.func = func

    def invoke(self, input):
        return self.func(input)
    
    def __or__(self, other):
        # returns a runnable
        def chained(input):
            return other.invoke(self.invoke(input))
        return Runnable(chained)